# **02477 — Assignment 1 Guide**

Tre modeller fra Assignment 1 der ikke er i eksamensguiderne men som kan dukke op til eksamen.

## Indhold
1. **Beta-Binomial model** — konjugat model for binære data (clicks, køb, ja/nej)
2. **Poisson-Gamma model** — konjugat model for tælledata (antal besøg, antal fejl)
3. **Linear Gaussian chain** — marginalisation og conditioning i kæder

---
**Mønsteret er det samme for alle tre:**
> Prior × Likelihood → Posterior (samme familie = konjugat)
> Posterior × ny likelihood → Posterior predictive

---

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, beta as beta_dist, gamma as gamma_dist
from scipy.special import beta as beta_fn, gammaln, comb
from scipy.stats import multivariate_normal as mvn
np.random.seed(42)
print('Imports OK.')

---
## Del 1 — Beta-Binomial Model

### Hvornår bruger du denne model?
Når du observerer **binære data** — succes/fiasko, køb/ikke-køb, klik/ikke-klik.

### Modellen
$$\theta \sim \text{Beta}(a_0, b_0) \quad \text{(prior)}$$
$$y|\theta \sim \text{Binomial}(N, \theta) \quad \text{(likelihood)}$$

### Konjugat opdatering
$$\boxed{p(\theta|y) = \text{Beta}(\theta|\underbrace{a_0 + y}_{a}, \underbrace{b_0 + N - y}_{b})}$$

**Intuition:** $a_0$ = pseudo-successer, $b_0$ = pseudo-fiaskoer. Data tilføjer $y$ successer og $N-y$ fiaskoer.

### Nøgleformler for Beta$(a,b)$
$$\mathbb{E}[\theta] = \frac{a}{a+b}, \quad \text{Var}[\theta] = \frac{ab}{(a+b)^2(a+b+1)}, \quad \text{Mode} = \frac{a-1}{a+b-2}$$

### Posterior predictive (Beta-Binomial PMF)
$$p(y^*=k|y) = \binom{N^*}{k}\frac{B(a+k,\; b+N^*-k)}{B(a,b)}$$
$$p(y^*>0|y) = 1 - p(y^*=0|y)$$

### Prior: uniform = Beta(1,1)
Uniform prior: $a_0=b_0=1$ giver $p(\theta)=1$ for $\theta\in[0,1]$
- Prior mean = 0.5
- 95% CI = [0.025, 0.975] (direkte fra uniform)

In [ ]:
# BETA-BINOMIAL
# INPUTS - skift disse
a0, b0 = 1, 1
N      = 115
y      = 4
N_star = 20

# POSTERIOR
a = a0 + y
b = b0 + N - y
post_mean = a / (a + b)
post_var  = (a * b) / ((a+b)**2 * (a+b+1))
CI_lo, CI_hi = beta_dist.interval(0.95, a=a, b=b)
MLE = y / N

print(f'Prior:     Beta({a0}, {b0})')
print(f'Data:      N={N}, y={y}  ->  MLE = {MLE:.4f}')
print(f'Posterior: Beta({a}, {b})')
print(f'  Mean:  E[theta|y] = a/(a+b) = {a}/{a+b} = {post_mean:.4f}')
print(f'  Std:   {np.sqrt(post_var):.4f}')
print(f'  95% CI: [{CI_lo:.4f}, {CI_hi:.4f}]')
print()

# POSTERIOR PREDICTIVE
def bb_pmf(k, a, b, N_star):
    return comb(N_star, k) * beta_fn(a+k, b+N_star-k) / beta_fn(a, b)

ks    = np.arange(0, N_star + 1)
probs = np.array([bb_pmf(k, a, b, N_star) for k in ks])

print(f'Posterior predictive p(y*=k|y) for N*={N_star}:')
print(f'  p(y*=0|y)  = {probs[0]:.4f}')
print(f'  p(y*=1|y)  = {probs[1]:.4f}')
print(f'  p(y*>0|y)  = 1 - p(y*=0|y) = {1-probs[0]:.4f}')
print(f'  E[y*|y]    = {np.sum(ks*probs):.4f}')
print(f'  Var[y*|y]  = {np.sum((ks-np.sum(ks*probs))**2 * probs):.4f}')
print(f'  Sum check: {probs.sum():.6f}  (skal vaere 1.0)')

# PLOT
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
theta_grid = np.linspace(0.001, 0.999, 500)
axes[0].plot(theta_grid, beta_dist.pdf(theta_grid, a0, b0), '--',
             label=f'Prior Beta({a0},{b0})')
axes[0].plot(theta_grid, beta_dist.pdf(theta_grid, a, b),
             label=f'Posterior Beta({a},{b})')
axes[0].axvline(post_mean, color='r', lw=1.5, linestyle=':', label=f'Post mean={post_mean:.3f}')
axes[0].set(xlabel='theta', title='Prior vs Posterior'); axes[0].legend()
axes[1].stem(ks, probs, basefmt=' ')
axes[1].set(xlabel='k', ylabel='p(y*=k|y)', title=f'Posterior predictive N*={N_star}')
plt.tight_layout(); plt.show()

### Hvad du skriver paa papiret

**Opgave: 'Bestem posterior p(theta|y)'**

> $$\log p(\theta|y) = \log p(y|\theta) + \log p(\theta) + \text{const}$$
> $$= y\log\theta + (N-y)\log(1-\theta) + (a_0-1)\log\theta + (b_0-1)\log(1-\theta) + \text{const}$$
> $$= (a_0+y-1)\log\theta + (b_0+N-y-1)\log(1-\theta) + \text{const}$$
>
> Dette matcher formen for $\log\text{Beta}(a,b) = (a-1)\log\theta + (b-1)\log(1-\theta) + \text{const}$, saa:
> $$\boxed{p(\theta|y) = \text{Beta}(\theta|a_0+y,\; b_0+N-y)}$$

---

---
## Del 2 — Poisson-Gamma Model

### Hvornaar bruger du denne model?
Naar du observerer **taelledata** — antal besog, antal fejl, antal haendelser per tidsenhed.

### Modellen
$$\lambda \sim \text{Gamma}(a_0, b_0) \quad \text{(prior, } b_0 \text{ er rate-parameteren)}$$
$$y_i|\lambda \sim \text{Poisson}(\lambda) \quad \text{i.i.d.}$$

### Konjugat opdatering
$$\boxed{p(\lambda|y) = \text{Gamma}\!\left(\lambda\;\Big|\;a_0 + \sum_{i=1}^N y_i,\; b_0 + N\right)}$$

**Intuition:** $a_0$ = pseudo-haendelser, $b_0$ = pseudo-tidsperioder.

### Noegleformler for Gamma$(a,b)$
$$\mathbb{E}[\lambda] = \frac{a}{b}, \quad \text{Var}[\lambda] = \frac{a}{b^2}, \quad \text{Mode} = \frac{a-1}{b}$$

### Log-pdf (til matching i derivationer)
$$\log p(\lambda|a,b) = (a-1)\log\lambda - b\lambda + \text{const}$$

### Posterior predictive
Brug Monte Carlo: sample $\lambda^{(i)}\sim\text{Gamma}(a,b)$, derefter $y^{*(i)}\sim\text{Poisson}(\lambda^{(i)})$

In [ ]:
# POISSON-GAMMA
# INPUTS - skift disse
a0, b0   = 1, 0.1
y_data   = np.array([7, 4, 8, 11, 12])
N        = len(y_data)

# POSTERIOR
a = a0 + np.sum(y_data)
b = b0 + N
post_mean = a / b
post_std  = np.sqrt(a / b**2)
CI_lo, CI_hi = gamma_dist.interval(0.95, a=a, scale=1/b)

print(f'Prior:     Gamma({a0}, {b0})')
print(f'Data:      N={N}, sum(y)={int(np.sum(y_data))}')
print(f'Posterior: Gamma({a}, {b})')
print(f'  E[lambda|y] = a/b = {a}/{b} = {post_mean:.4f}')
print(f'  Std:         {post_std:.4f}')
print(f'  95% CI:      [{CI_lo:.4f}, {CI_hi:.4f}]')
print()

# POSTERIOR PREDICTIVE via Monte Carlo
np.random.seed(0); S = 100_000
lam_samples = np.random.gamma(shape=a, scale=1/b, size=S)
ystar_samps = np.random.poisson(lam_samples)
print(f'Posterior predictive p(y*|y):')
print(f'  E[y*|y]    = {np.mean(ystar_samps):.4f}  (approx = E[lambda|y] = {post_mean:.4f})')
print(f'  Var[y*|y]  = {np.var(ystar_samps):.4f}')
print(f'  95% CI:    [{np.percentile(ystar_samps,2.5):.1f}, {np.percentile(ystar_samps,97.5):.1f}]')
print(f'  p(y*=0|y)  = {np.mean(ystar_samps==0):.4f}')
print(f'  p(y*>10|y) = {np.mean(ystar_samps>10):.4f}')

# PLOT
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
lam_grid = np.linspace(0.001, 30, 500)
axes[0].plot(lam_grid, gamma_dist.pdf(lam_grid, a=a0, scale=1/b0), '--',
             label=f'Prior Gamma({a0},{b0})')
axes[0].plot(lam_grid, gamma_dist.pdf(lam_grid, a=a, scale=1/b),
             label=f'Posterior Gamma({a},{b:.1f})')
axes[0].axvline(post_mean, color='r', lw=1.5, linestyle=':', label=f'Post mean={post_mean:.2f}')
axes[0].set(xlabel='lambda', title='Prior vs Posterior'); axes[0].legend()
axes[1].hist(ystar_samps, bins=range(0,31), density=True, alpha=0.7, label='MC posterior predictive')
axes[1].set(xlabel='y*', title='Posterior predictive p(y*|y)'); axes[1].legend()
plt.tight_layout(); plt.show()

### Hvad du skriver paa papiret

**Opgave: 'Vis at posterior er en Gamma-fordeling'**

> $$\log p(\lambda|y) = \sum_{i=1}^N [y_i\log\lambda - \lambda] + (a_0-1)\log\lambda - b_0\lambda + \text{const}$$
> $$= \left(a_0 + \sum_i y_i - 1\right)\log\lambda - (b_0+N)\lambda + \text{const}$$
>
> Koefficient-matching med $(a-1)\log\lambda - b\lambda + \text{const}$ giver:
> $$\boxed{p(\lambda|y) = \text{Gamma}\!\left(\lambda\;\Big|\;a_0+\sum_i y_i,\; b_0+N\right)}$$

**Noegle-trick:** Saml alle termer med $\log\lambda$ (giver $a-1$) og termer med $\lambda$ (giver $-b$). Afloes $a$ og $b$ direkte.

---

---
## Del 3 — Linear Gaussian Chain

### Modellen fra Assignment 1
$$z_1 \sim \mathcal{N}(0, vI), \quad z_2|z_1 \sim \mathcal{N}(z_1, vI), \quad y|z_2 \sim \mathcal{N}(a^Tz_2, \sigma^2)$$

### Den generelle marginaliseringsformel (Murphy eq. 3.38)

<span style="color: blue;">
$$\int \mathcal{N}(y|Wz+b,\Sigma_y)\,\mathcal{N}(z|\mu_z,\Sigma_z)\,dz = \mathcal{N}(y|W\mu_z+b,\;\Sigma_y+W\Sigma_z W^T)$$
</span>

**Husk:** Mean propagerer lineaert. Kovarianserne **adderer**.

### De fire opgaver og svar

**Task 2.1 — $p(y)$:** To-trins marginalisation.

*Trin 1* — Find $p(z_2)$ ved at marginalisere $z_1$ (saet $W=I$, $b=0$, $\mu_z=0$, $\Sigma_z=vI$, $\Sigma_y=vI$):
$$p(z_2) = \mathcal{N}(z_2|0,\; 2vI)$$

*Trin 2* — Find $p(y)$ ved at marginalisere $z_2$ (saet $W=a^T$, $b=0$, $\mu_z=0$, $\Sigma_z=2vI$, $\Sigma_y=\sigma^2$):
$$\boxed{p(y) = \mathcal{N}(y|0,\; \sigma^2+2va^Ta)}$$

---

**Task 2.2 — $p(y,z_2|z_1)$:** Produktregel + betinget uafhaengighed.
$$p(y,z_2|z_1) = p(y|z_2)\,p(z_2|z_1) = \mathcal{N}(y|a^Tz_2,\sigma^2)\,\mathcal{N}(z_2|z_1,vI)$$

---

**Task 2.3 — $p(y|z_1)$:** Marginaliseer $z_2$ fra Task 2.2 (saet $W=a^T$, $\mu_z=z_1$, $\Sigma_z=vI$):
$$\boxed{p(y|z_1) = \mathcal{N}(y|a^Tz_1,\; \sigma^2+va^Ta)}$$

---

**Task 2.4 — $p(z_1|y)$:** Bayes' regel + Murphy (3.37).

Likelihood: $p(y|z_1)=\mathcal{N}(y|a^Tz_1, \sigma^2+va^Ta)$. Prior: $p(z_1)=\mathcal{N}(z_1|0,vI)$.

$$\Sigma_{z_1|y} = \left(\frac{1}{v}I + \frac{a\,a^T}{\sigma^2+va^Ta}\right)^{-1}, \qquad \mu_{z_1|y} = \Sigma_{z_1|y}\,\frac{a}{\sigma^2+va^Ta}\,y$$

In [ ]:
# LINEAR GAUSSIAN CHAIN - numerisk verifikation
# INPUTS
v      = 1.0
sigma2 = 0.5
a_vec  = np.array([1.0, 2.0])

# p(z2) = N(0, 2v*I)
print(f'p(z2) = N(0, 2v*I) = N(0, {2*v}*I)')

# p(y) = N(0, sigma^2 + 2v*a^T*a)
var_y = sigma2 + 2*v * a_vec @ a_vec
print(f'p(y)  = N(0, {var_y:.4f})  [sigma^2 + 2v*||a||^2 = {sigma2}+2*{v}*{a_vec@a_vec}]')
print()

# p(y|z1) = N(a^T*z1, sigma^2 + v*a^T*a)
var_y_z1 = sigma2 + v * a_vec @ a_vec
print(f'p(y|z1) = N(a^T*z1, {var_y_z1:.4f})  [sigma^2 + v*||a||^2]')
print()

# p(z1|y) via Murphy (3.37)
S_inv = (1/v)*np.eye(2) + np.outer(a_vec, a_vec)/var_y_z1
S_post = np.linalg.inv(S_inv)
y_obs  = 3.0
mu_post = S_post @ a_vec / var_y_z1 * y_obs
print(f'p(z1|y): Sigma_z1|y ='); print(np.round(S_post, 4))
print(f'For y={y_obs}: mu_z1|y = {np.round(mu_post,4)}')
print()

# Monte Carlo verifikation
np.random.seed(0); S = 200_000
z1_mc = np.random.multivariate_normal(np.zeros(2), v*np.eye(2), S)
z2_mc = z1_mc + np.random.multivariate_normal(np.zeros(2), v*np.eye(2), S)
y_mc  = z2_mc @ a_vec + np.random.normal(0, np.sqrt(sigma2), S)
print(f'MC check p(y): mean={np.mean(y_mc):.3f} (expect 0), var={np.var(y_mc):.4f} (expect {var_y:.4f})')

---
## Oversigt — De tre modeller

| Model | Prior | Likelihood | Posterior | Update-regel |
|-------|-------|------------|-----------|---------------|
| **Beta-Binomial** | Beta$(a_0,b_0)$ | Binomial$(N,\theta)$ | Beta$(a_0+y,\; b_0+N-y)$ | Tael successer og fiaskoer |
| **Poisson-Gamma** | Gamma$(a_0,b_0)$ | Poisson$(\lambda)$ | Gamma$(a_0+\sum y_i,\; b_0+N)$ | Tael haendelser og observationer |
| **Linear Gaussian** | $\mathcal{N}(\mu_z,\Sigma_z)$ | $\mathcal{N}(Wz+b,\Sigma_y)$ | Murphy (3.37) | Praecisioner adderer |

---

## Typiske eksamensformauleringer

| Formulering | Hvad du goer |
|-------------|---------------|
| *'Bestem posterior p(theta\|y)'* | Skriv log-posterior, saml termer, matche koefficienter |
| *'Beregn posterior mean'* | $a/(a+b)$ for Beta, $a/b$ for Gamma |
| *'95% credibility interval'* | `beta.interval(0.95, a, b)` eller `gamma.interval(0.95, a, scale=1/b)` |
| *'Posterior predictive p(y*\|y)'* | Beta-Binomial PMF eller MC med Poisson |
| *'p(y*>0\|y)'* | $1 - p(y^*=0|y)$ |
| *'Vis at posterior er Gamma/Beta'* | Log-posterior + koefficient-matching |
| *'Bestem p(y)'* | Marginaliser med Murphy (3.38) i trin |
| *'Bestem p(x\|y)'* | Bayes + Murphy (3.37) |

---

## Quick copy-paste kode

```python
# Beta-Binomial
from scipy.stats import beta
a_post = a0 + y
b_post = b0 + N - y
mean   = a_post / (a_post + b_post)
lo, hi = beta.interval(0.95, a=a_post, b=b_post)

# Poisson-Gamma
from scipy.stats import gamma
a_post = a0 + sum(y_data)
b_post = b0 + N
mean   = a_post / b_post
lo, hi = gamma.interval(0.95, a=a_post, scale=1/b_post)

# Beta-Binomial posterior predictive
from scipy.special import beta as beta_fn, comb
def bb_pmf(k, a, b, N_star):
    return comb(N_star,k) * beta_fn(a+k, b+N_star-k) / beta_fn(a,b)
p_at_least_one = 1 - bb_pmf(0, a_post, b_post, N_star)

# Poisson-Gamma posterior predictive (MC)
lam_s = np.random.gamma(a_post, 1/b_post, size=100_000)
y_star = np.random.poisson(lam_s)
# Brug np.mean, np.percentile, np.mean(y_star > k) osv.
```